Перевiрити модель, отриману в результатi виконання завдання 7 (вiльну вiд
автокореляцiї), на гетероскедастичнiсть залишкiв за допомогою тесту 
Гольдфельда—Квандта (послiдовно для кожної незалежної змiнної).
Оцiнивши σi, побудувати матрицю перетворень T^H. Перетворити початковi
данi i розрахувати методом 1МНК коефiцiєнти нової моделi. Показати, що у
побудованiй моделi гетероскедастичнiсть вiдсутня.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

data = {
    'Країна': [
        'Австралія', 'Австрія', 'Аргентина', 'Бангладеш', 'Бельгія', 'Білорусь', 'Бразилія', 'Буркіна-Фасо', 
        'Велика Британія', "В'єтнам", 'Гаїті', 'Гондурас', 'Гонконг', 'Ефіопія', 'Єгипет', 'Замбія', 'Індія', 
        'Ірландія', 'Іспанія', 'Італія', 'Канада', 'Китай', 'Колумбія', 'Коста-Рика', 'Куба', 'Малайзія', 
        'Марокко', 'Мексика', 'Нідерланди', 'Німеччина', 'Нова Зеландія', 'Норвегія', 'ОАЕ', 'ПАР', 
        'Південна Корея', 'Північна Корея', 'Польща', 'Португалія', 'Росія', 'Саудівська Аравія', 'Сінгапур', 
        'США', 'Таїланд', 'Туреччина', 'Україна', 'Філіппіни', 'Фінляндія', 'Франція', 'Чилі', 'Швейцарія', 
        'Швеція', 'Японія'
    ],
    'X0': np.ones(52),
    'Y1': [74, 73, 68, 53, 73, 66, 57, 47, 74, 63, 43, 65, 75, 51, 60, 44, 58, 73, 74, 74, 74, 67, 69, 76, 74, 66, 66, 69, 75, 73, 73, 74, 70, 62, 68, 67, 69, 71, 64, 66, 73, 73, 65, 69, 65, 63, 72, 74, 71, 75, 75, 76],
    'X1': [17800, 8000, 33900, 125000, 10100, 10300, 156600, 10000, 58400, 73100, 6500, 5600, 5800, 55200, 60000, 9100, 911600, 3600, 39200, 58100, 29100, 1205200, 35600, 3300, 11100, 19500, 28600, 91800, 15400, 81200, 3524, 4300, 2800, 43900, 45000, 23100, 38600, 10500, 149200, 18000, 2900, 260800, 59400, 62200, 51800, 69800, 5100, 58000, 14000, 7000, 8800, 125500],
    'X2': [15, 12, 20, 35, 12, 13, 21, 47, 13, 27, 40, 35, 13, 45, 29, 46, 29, 14, 11, 11, 14, 21, 24, 26, 17, 29, 29, 28, 13, 11, 16, 13, 28, 34, 16, 24, 14, 12, 13, 38, 16, 15, 19, 26, 12, 27, 13, 13, 23, 12, 14, 11],
    'X3': [8, 11, 9, 11, 11, 11, 9, 18, 11, 8, 19, 6, 6, 14, 9, 18, 10, 9, 9, 10, 8, 7, 6, 4, 7, 5, 6, 5, 9, 11, 8, 10, 3, 8, 6, 6, 10, 10, 11, 6, 6, 9, 6, 6, 13, 7, 10, 9, 6, 9, 11, 7],
    'X4': [16848, 18396, 3408, 202, 17912, 6500, 2354, 357, 15974, 230, 383, 1030, 14641, 122, 748, 573, 275, 12170, 13047, 17500, 19904, 377, 1538, 2031, 1382, 2995, 1062, 3604, 17245, 17539, 14381, 17755, 14193, 3128, 6627, 1000, 4429, 9000, 6680, 6651, 14990, 23474, 1800, 3721, 2340, 867, 15877, 18944, 2591, 22384, 16900, 19860],
    'X5': [85, 58, 86, 16, 96, 65, 75, 15, 89, 20, 29, 44, 94, 12, 44, 42, 26, 57, 78, 69, 77, 26, 70, 47, 74, 43, 46, 73, 89, 85, 84, 75, 81, 49, 72, 60, 62, 34, 74, 77, 100, 75, 22, 61, 67, 43, 60, 73, 85, 62, 84, 77]
}
df = pd.DataFrame(data)

In [ ]:
df_work = df.copy()
y_col = 'Y1'
x_cols = ['X0', 'X1', 'X2', 'X3', 'X4', 'X5']
test_vars = ['X1', 'X2', 'X3', 'X4', 'X5']
remaining_cols = [c for c in df_work.columns if c not in x_cols]

# Рівні значущості як у прикладі 8.1:
# alpha_gq = 0.05 (ймовірність 90% для тесту Гольдфельда — Квандта)
# alpha_aux = 0.05 (ймовірність 95% для перевірки допоміжної регресії u_t^2)
alpha_gq = 0.05
alpha_aux = 0.05

T = len(df_work)  # 52 спостереження
N = len(x_cols)  # 6 параметрів моделі

In [ ]:
def run_goldfeld_quandt(
    data_df, y_name, x_names, check_vars, alpha=0.05, stage_title='', verbose=True
):
  """Виконує тест Гольдфельда — Квандта (Алгоритм 8.1) для всіх змінних із check_vars."""
  n_obs = len(data_df)
  n_params = len(x_names)
  m = round((4 / 15) * n_obs)  # round(4/15 * 52) = 14
  t_rem = n_obs - m
  t1 = t_rem // 2  # 19
  t2 = t_rem - t1  # 19
  df1 = t1 - n_params  # 13
  df2 = t2 - n_params  # 13

  print('\n' + '=' * 80)
  print(f'ТЕСТ ГОЛЬДФЕЛЬДА — КВАНДТА: {stage_title}')
  print(
      f'T = {n_obs}, N = {n_params}, m = {m}, T1 = {t1}, T2 = {t2}, '
      f'ступені вільності = ({df2}, {df1})'
  )
  print('=' * 80)

  results = {}
  for var in check_vars:
    df_sorted = data_df.sort_values(by=var, ascending=True).reset_index(
        drop=True
    )
    g1 = df_sorted.iloc[:t1]
    g2 = df_sorted.iloc[t1 + m :]

    Y1, X1 = g1[y_name].to_numpy(float), g1[x_names].to_numpy(float)
    Y2, X2 = g2[y_name].to_numpy(float), g2[x_names].to_numpy(float)

    b1, _, _, _ = np.linalg.lstsq(X1, Y1, rcond=None)
    u1 = Y1 - X1 @ b1
    rss1 = np.sum(u1**2)
    s1_sq = rss1 / df1

    b2, _, _, _ = np.linalg.lstsq(X2, Y2, rcond=None)
    u2 = Y2 - X2 @ b2
    rss2 = np.sum(u2**2)
    s2_sq = rss2 / df2

    f_stat = s2_sq / s1_sq
    results[var] = {
        'g1': g1,
        'g2': g2,
        'b1': b1,
        'b2': b2,
        'u1': u1,
        'u2': u2,
        'rss1': rss1,
        'rss2': rss2,
        's1_sq': s1_sq,
        's2_sq': s2_sq,
        'F_stat': f_stat,
    }

    if verbose:
      print(f'\n--- Змінна {var} ---')
      print(
          '  Модель 1-ї групи (y^1): '
          + ' + '.join(f'{coef:.6f}*{col}' for coef, col in zip(b1, x_names))
      )
      print(
          '  Модель 2-ї групи (y^2): '
          + ' + '.join(f'{coef:.6f}*{col}' for coef, col in zip(b2, x_names))
      )
      print(f'  sum(u1^2) = {rss1:.6f}, s1^2 = {s1_sq:.6f}')
      print(f'  sum(u2^2) = {rss2:.6f}, s2^2 = {s2_sq:.6f}')
      print(f'  F_СТ^({var}) = {f_stat:.4f}')

  print('\nЗведені значення F_СТ:')
  for var in check_vars:
    print(f"  F_СТ^({var}) = {results[var]['F_stat']:.4f}")

  best_var = max(results, key=lambda k: results[k]['F_stat'])
  f_max = results[best_var]['F_stat']
  f_crit = stats.f.ppf(1 - alpha, dfn=df2, dfd=df1)

  print(
      f'\nМаксимальне F_СТ^({best_var}) = {f_max:.4f} | '
      f'F_КР = F({alpha}; {df2}; {df1}) = {f_crit:.4f}'
  )
  has_hetero = f_max > f_crit
  if has_hetero:
    print(
        f'Висновок: F_СТ ({f_max:.4f}) > F_КР ({f_crit:.4f}) -> '
        f'гетероскедастичність ІСНУЄ з ймовірністю {(1 - alpha)*100:.0f}%.'
    )
  else:
    print(
        f'Висновок: усі F_СТ < F_КР ({f_crit:.4f}) -> '
        f'гетероскедастичність ВІДСУТНЯ з ймовірністю {(1 - alpha)*100:.0f}%.'
    )

  return results, best_var, has_hetero

In [11]:
# ==============================================================================
# ЕТАП 1. ПОЧАТКОВА МОДЕЛЬ ТА ПЕРЕВІРКА НА ГЕТЕРОСКЕДАСТИЧНІСТЬ
# ==============================================================================
Y = df_work[y_col].to_numpy(dtype=float)
X = df_work[x_cols].to_numpy(dtype=float)

# Оцінювання початкової моделі на всіх T=52 спостереженнях методом 1МНК
b_init, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
y_hat_init = X @ b_init
u_init = Y - y_hat_init
u_sq = u_init**2

print('=' * 80)
print('ЕТАП 1. ПОЧАТКОВА РЕГРЕСІЙНА МОДЕЛЬ (ВСІ 52 СПОСТЕРЕЖЕННЯ)')
print('=' * 80)
print(
    'y_hat = '
    + ' + '.join(f'{coef:.6f}*{col}' for coef, col in zip(b_init, x_cols))
)

# Тест Гольдфельда — Квандта для початкової моделі
gq_init_res, best_var, _ = run_goldfeld_quandt(
    df_work,
    y_col,
    x_cols,
    test_vars,
    alpha=alpha_gq,
    stage_title='ПОЧАТКОВА МОДЕЛЬ',
    verbose=True,
)

# Детальний вивід залишків для змінної з найбільшим F_СТ
best_pack = gq_init_res[best_var]
print(f'\nВектор залишків u^1 (перша група за {best_var}):')
print(np.round(best_pack['u1'], 4))
print(f'Вектор залишків u^2 (друга група за {best_var}):')
print(np.round(best_pack['u2'], 4))

ЕТАП 1. ПОЧАТКОВА РЕГРЕСІЙНА МОДЕЛЬ (ВСІ 52 СПОСТЕРЕЖЕННЯ)
y_hat = 82.170225*X0 + -0.000002*X1 + -0.374771*X2 + -1.150075*X3 + 0.000326*X4 + 0.015112*X5

ТЕСТ ГОЛЬДФЕЛЬДА — КВАНДТА: ПОЧАТКОВА МОДЕЛЬ
T = 52, N = 6, m = 14, T1 = 19, T2 = 19, ступені вільності = (13, 13)

--- Змінна X1 ---
  Модель 1-ї групи (y^1): 89.525166*X0 + 0.000233*X1 + -0.459979*X2 + -1.384762*X3 + 0.000207*X4 + -0.031566*X5
  Модель 2-ї групи (y^2): 77.191877*X0 + 0.000000*X1 + -0.293449*X2 + -1.053908*X3 + 0.000461*X4 + 0.017740*X5
  sum(u1^2) = 95.881910, s1^2 = 7.375532
  sum(u2^2) = 118.059980, s2^2 = 9.081537
  F_СТ^(X1) = 1.2313

--- Змінна X2 ---
  Модель 1-ї групи (y^1): 79.064923*X0 + -0.000018*X1 + -0.403395*X2 + -0.793176*X3 + 0.000393*X4 + 0.014125*X5
  Модель 2-ї групи (y^2): 77.858173*X0 + -0.000002*X1 + -0.105868*X2 + -1.661876*X3 + -0.000241*X4 + 0.049449*X5
  sum(u1^2) = 27.707414, s1^2 = 2.131340
  sum(u2^2) = 74.757272, s2^2 = 5.750559
  F_СТ^(X2) = 2.6981

--- Змінна X3 ---
  Модель 1-ї групи 

In [ ]:
# ==============================================================================
# ЕТАП 2. ТРЕТІЙ СПОСІБ РОЗРАХУНКУ sigma_i (ДОПОМІЖНА РЕГРЕСІЯ u_t^2 НА X)
# ==============================================================================
print('\n' + '=' * 80)
print('ЕТАП 2. ДОПОМІЖНА РЕГРЕСІЯ ДЛЯ u_t^2 (ФОРМУЛА (*))')
print('=' * 80)

# Оцінка коефіцієнтів a_1..a_6 моделі: u_t^2 = a_1*x_1t + ... + a_6*x_6t + eps_t
a_hat, _, _, _ = np.linalg.lstsq(X, u_sq, rcond=None)
u_sq_hat = X @ a_hat
eps_hat = u_sq - u_sq_hat

print(
    'u_hat^2 = '
    + ' + '.join(f'{coef:.6f}*{col}' for coef, col in zip(a_hat, x_cols))
)

# Перевірка значущості допоміжної регресії (*)
# Формула з підручника: F_СТ = (u_sq_hat^T * u_sq_hat * (T - N)) / (eps_hat^T * eps_hat * (N - 1))
num_textbook = (u_sq_hat @ u_sq_hat) * (T - N)
den_textbook = (eps_hat @ eps_hat) * (N - 1)
F_aux_textbook = num_textbook / den_textbook

# Центрований варіант F-статистики (для довідки)
# ess_centered = np.sum((u_sq_hat - np.mean(u_sq)) ** 2)
# F_aux_centered = (ess_centered / (N - 1)) / ((eps_hat @ eps_hat) / (T - N))

F_aux_crit = stats.f.ppf(1 - alpha_aux, dfn=N - 1, dfd=T - N)

print(f'\nF_СТ = {F_aux_textbook:.4f}')
# print(f'F_СТ (центрований F-критерій) = {F_aux_centered:.4f}')
print(f'F_КР = F({alpha_aux}; {N - 1}; {T - N}) = {F_aux_crit:.4f}')

if F_aux_textbook > F_aux_crit:
  print(
      f'Висновок: F_СТ ({F_aux_textbook:.4f}) > F_КР ({F_aux_crit:.4f}) -> '
      f'модель (*) значимо відрізняється від нуля з ймовірністю {(1 - alpha_aux)*100:.0f}%.'
  )
else:
  print(
      f'Висновок: F_СТ ({F_aux_textbook:.4f}) <= F_КР ({F_aux_crit:.4f}) -> '
      'модель (*) не значимо відрізняється від нуля.'
  )


ЕТАП 2. ДОПОМІЖНА РЕГРЕСІЯ ДЛЯ u_t^2 (ФОРМУЛА (*))
u_hat^2 = 10.582600*X0 + -0.000000*X1 + -0.121073*X2 + -0.236225*X3 + -0.000648*X4 + 0.095673*X5

F_СТ (за формулою підручника) = 3.5928
F_СТ (центрований F-критерій) = 0.9141
F_КР = F(0.05; 5; 46) = 2.4174
Висновок: F_СТ (3.5928) > F_КР (2.4174) -> модель (*) значуща з ймовірністю 95%.


In [13]:
# ==============================================================================
# ЕТАП 3. РОЗРАХУНОК sigma_i, МАТРИЦІ ПЕРЕТВОРЕНЬ T^H ТА НОВОЇ МОДЕЛІ (**)
# ==============================================================================
print('\n' + '=' * 80)
print('ЕТАП 3. МАТРИЦЯ ПЕРЕТВОРЕНЬ T^H ТА УСУНЕННЯ ГЕТЕРОСКЕДАСТИЧНОСТІ')
print('=' * 80)

sigma = np.sqrt(np.abs(u_sq_hat))
inv_sigma = 1.0 / sigma
T_H = np.diag(inv_sigma)  # Діагональна матриця перетворень розміром 52 x 52

# Таблиця значень u_t, u_t^2, u_hat^2, sigma_i та 1/sigma_i для всіх 52 спостережень
transform_table = pd.DataFrame({
    'i': np.arange(1, T + 1),
    'u_i': u_init,
    'u_i^2': u_sq,
    'u_hat_i^2': u_sq_hat,
    'sigma_i': sigma,
    '1/sigma_i (діаг. T^H)': inv_sigma,
})
print('\nПокрокова таблиця розрахунку sigma_i та діагоналі матриці T^H:')
print(transform_table.to_string(index=False))

# Перетворення початкових даних: [T^H * Y | T^H * X] = [Y^1 | X^1]
Y_star = T_H @ Y
X_star = T_H @ X

df_transformed = pd.DataFrame(X_star, columns=x_cols)
df_transformed.insert(0, y_col, Y_star)

print('\nПерші 10 рядків перетворених даних [Y^1 | X^1] = [T^H Y | T^H X]:')
print(df_transformed.head(10).to_string(index=False))

# Оцінювання скоригованої регресійної моделі (**) методом 1МНК
b_gls, _, _, _ = np.linalg.lstsq(X_star, Y_star, rcond=None)

print('\n' + '-' * 80)
print('РЕГРЕСІЙНА МОДЕЛЬ (**), ЗВІЛЬНЕНА ВІД ГЕТЕРОСКЕДАСТИЧНОСТІ:')
print(
    'y_hat = '
    + ' + '.join(f'{coef:.6f}*{col}' for coef, col in zip(b_gls, x_cols))
)
print('-' * 80)


ЕТАП 3. МАТРИЦЯ ПЕРЕТВОРЕНЬ T^H ТА УСУНЕННЯ ГЕТЕРОСКЕДАСТИЧНОСТІ

Покрокова таблиця розрахунку sigma_i та діагоналі матриці T^H:
 i       u_i     u_i^2  u_hat_i^2  sigma_i  1/sigma_i (діаг. T^H)
 1 -0.096235  0.009261   4.086190 2.021433               0.494699
 2  1.113381  1.239617   0.159449 0.399311               2.504314
 3  1.330189  1.769404  12.039048 3.469733               0.288207
 4 -3.466104 12.013878   5.086583 2.255345               0.443391
 5  0.701172  0.491643   4.107565 2.026713               0.493410
 6 -1.730856  2.995862   8.413268 2.900563               0.344761
 7 -8.545277 73.021751  11.489572 3.389627               0.295018
 8  2.821674  7.961845   1.839135 1.356147               0.737383
 9  2.908498  8.459360   4.549084 2.132858               0.468854
10 -0.085407  0.007294   7.153265 2.674559               0.373893
11 -2.878534  8.285957   3.774683 1.942854               0.514707
12  1.857049  3.448630   8.467381 2.909876               0.343657
13 -1.585104

In [14]:
# ==============================================================================
# ЕТАП 4. ПОВТОРНИЙ ТЕСТ ГОЛЬДФЕЛЬДА — КВАНДТА ДЛЯ МОДЕЛІ (**)
# ==============================================================================
run_goldfeld_quandt(
    df_transformed,
    y_col,
    x_cols,
    test_vars,
    alpha=alpha_gq,
    stage_title='СКОРИГОВАНА МОДЕЛЬ (**) ПІСЛЯ ПЕРЕТВОРЕННЯ T^H',
    verbose=False,
)


ТЕСТ ГОЛЬДФЕЛЬДА — КВАНДТА: СКОРИГОВАНА МОДЕЛЬ (**) ПІСЛЯ ПЕРЕТВОРЕННЯ T^H
T = 52, N = 6, m = 14, T1 = 19, T2 = 19, ступені вільності = (13, 13)

Зведені значення F_СТ:
  F_СТ^(X1) = 1.0853
  F_СТ^(X2) = 1.5675
  F_СТ^(X3) = 1.9982
  F_СТ^(X4) = 0.1454
  F_СТ^(X5) = 0.2334

Максимальне F_СТ^(X3) = 1.9982 | F_КР = F(0.05; 13; 13) = 2.5769
Висновок: усі F_СТ < F_КР (2.5769) -> гетероскедастичність ВІДСУТНЯ з ймовірністю 95%.


({'X1': {'g1':            Y1        X0           X1         X2        X3            X4  \
   0   24.441011  0.321592  1061.254443   8.361399  1.286369    653.153871   
   1   27.428447  0.375732  1089.623224   6.011714  2.254393   5632.224870   
   2   31.187363  0.445534  1247.494528  12.474945  1.336601   6323.460655   
   3   31.199956  0.427397  1506.145804   6.838346  3.419173   6146.391262   
   4   35.083921  0.480602  1730.165955   6.728423  4.325415   5848.922132   
   5   22.337718  0.343657  1924.480361  12.028002  2.061943    353.966924   
   6   28.152352  0.375365  2177.115215   4.879741  2.252188   5495.714459   
   7   48.603683  0.656807  2824.268093   8.538485  6.568065  11661.599996   
   8   20.484541  0.276818  3072.681097   4.705908  1.937727    382.562637   
   9   22.132384  0.514707  3345.592936  20.588264  9.779426    197.132630   
   10  49.694705  0.690204  3520.041615   8.972655  6.902042  10958.372692   
   11  22.754201  0.344761  3551.034348   4.481888  